# Gider Talebi Analizi

Bu not defteri, yerel makbuz görüntülerinden seyahat giderlerini işlemek, bir gider talep e-postası oluşturmak ve gider verilerini pasta grafiği ile görselleştirmek için eklentiler kullanan ajanların nasıl oluşturulacağını gösterir. Ajanlar, görev bağlamına göre dinamik olarak işlevleri seçer.

Adımlar:
1. OCR Ajanı, yerel makbuz görüntüsünü işler ve seyahat gideri verilerini çıkarır.
2. E-posta Ajanı, bir gider talep e-postası oluşturur.

### Seyahat gideri senaryosu örneği:
Bir iş toplantısı için başka bir şehre seyahat eden bir çalışan olduğunuzu hayal edin. Şirketiniz tüm makul seyahatle ilgili giderlerin geri ödenmesi politikasına sahiptir. İşte olası seyahat giderlerinin dökümü:
- Ulaşım:
Ev şehrinizden varış şehrine gidiş-dönüş uçak bileti.
Havaalanına ve havaalanından taksi veya yolculuk çağırma hizmetleri.
Varış şehrindeki yerel ulaşım (toplu taşıma, kiralık araçlar veya taksiler gibi).

- Konaklama:
Toplantı yerinin yakınında orta sınıf bir iş otelinde üç gece konaklama.

- Yemekler:
Şirketin günlük ücret politikasına göre kahvaltı, öğle ve akşam yemekleri için günlük yemek harcırahı.

- Çeşitli Giderler:
Havaalanı otopark ücretleri.
Otelde internet erişim ücretleri.
Bahşişler veya küçük servis ücretleri.

- Belgeler:
Tüm makbuzları (uçuşlar, taksiler, otel, yemekler vb.) ve tamamlanmış bir gider raporunu geri ödeme için sunarsınız.


## Gerekli kütüphaneleri içe aktarın

Defter için gerekli kütüphane ve modülleri içe aktarın.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Gider Modellerini Tanımla

 Bireysel giderler için bir Pydantic modeli oluşturun ve bir kullanıcı sorgusunu yapılandırılmış gider verisine dönüştürmek için bir ExpenseFormatter sınıfı oluşturun.

 Her gider şu formatta temsil edilecektir:
 `{'date': '07-Mar-2025', 'description': 'varış noktasına uçuş', 'amount': 675.99, 'category': 'Ulaşım'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Araç Tanımlama - E-posta Oluşturma

Bir gider talebi göndermek için e-posta oluşturan bir araç fonksiyonu oluşturun.
- Bu araç Microsoft Agent Framework'ten `@tool` dekoratörünü kullanır.
- Giderlerin toplam tutarını hesaplar ve detayları e-posta gövdesi olarak formatlar.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Makbuz Görüntülerinden Seyahat Giderlerini Çıkarmak için Araç

Fiş görüntülerinden seyahat giderlerini çıkarmak için bir araç işlevi oluşturun.
- Bu araç, Microsoft Agent Framework'ten `@tool` dekoratörünü kullanır.
- Fiş görüntüsünü okur, base64 olarak kodlar ve ajan tarafından analiz edilmek üzere veri URI'sını döndürür.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Giderlerin İşlenmesi

Ajanları tanımlayın ve onları `WorkflowBuilder` kullanarak sıralı bir iş akışına bağlayın.
- OCR ajanı, `load_receipt_image` aracını kullanarak makbuz görüntüsünden yapılandırılmış gider verilerini çıkarır.
- E-posta ajanı, çıkarılan verileri alır ve `generate_expense_email` aracını kullanarak profesyonel bir gider talep e-postası oluşturur.
- `WorkflowBuilder` ile `add_edge` kullanarak sıralı bir boru hattı oluşturur: OCR Ajanı → E-posta Ajanı.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Ana fonksiyon

Sıralı iş akışını oluşturun ve fiş görüntüsünü işlemek ve gider talep e-postasını oluşturmak için çalıştırın.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilindeki haliyle yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilmektedir. Bu çevirinin kullanımı nedeniyle oluşabilecek her türlü yanlış anlama veya yorumlama için sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
